# Лаборатори 5: Пропозицийн логик
Үнэний хүснэгт, CNF, резолюц ба DPLL хэрэгжүүлэлт

In [1]:
import itertools
import random

## Алхам 1: Өгүүлбэрийг илэрхийлэх

In [2]:
def Sym(n):
    return ('sym', n)

def Not(a):
    return ('not', a)

def And(a, b):
    return ('and', a, b)

def Or(a, b):
    return ('or', a, b)

def Imp(a, b):
    return ('imp', a, b)

def Iff(a, b):
    return ('iff', a, b)

def symbols(s):
    if s[0] == 'sym':
        return {s[1]}
    result = set()
    for part in s[1:]:
        result |= symbols(part)
    return result

OPS = {'and': '&', 'or': '|', 'imp': '=>', 'iff': '<=>'}

def to_str(s):
    if s[0] == 'sym':
        return s[1]
    if s[0] == 'not':
        return '~' + to_str(s[1])
    return '(' + to_str(s[1]) + ' ' + OPS[s[0]] + ' ' + to_str(s[2]) + ')'

In [4]:
P, Q, R = Sym('P'), Sym('Q'), Sym('R')
s1 = Imp(And(P, Q), R)
print(to_str(s1))
print(symbols(s1) == {'P', 'Q', 'R'})

s2 = Iff(Not(P), Or(Q, R))
s3 = Not(And(Imp(P, Q), Not(R)))
for s in (s2, s3):
    print(to_str(s), sorted(symbols(s)))

((P & Q) => R)
True
(~P <=> (Q | R)) ['P', 'Q', 'R']
~((P => Q) & ~R) ['P', 'Q', 'R']


## Алхам 2: Семантик

In [5]:
def pl_true(s, model):
    op = s[0]
    if op == 'sym':
        return model[s[1]]
    if op == 'not':
        return not pl_true(s[1], model)
    a = pl_true(s[1], model)
    b = pl_true(s[2], model)
    if op == 'and':
        return a and b
    if op == 'or':
        return a or b
    if op == 'imp':
        return (not a) or b
    return a == b

In [5]:
m1 = {'P': True, 'Q': False, 'R': True}
print('(P & Q) => R :', pl_true(s1, m1))
print('P <=> Q      :', pl_true(Iff(P, Q), m1))
print('~P | Q       :', pl_true(Or(Not(P), Q), m1))

(P & Q) => R : True
P <=> Q      : False
~P | Q       : False


## Алхам 3: Үнэний хүснэгт ба логик дүгнэлт

In [6]:
def all_models(syms):
    syms = sorted(syms)
    for values in itertools.product([True, False], repeat=len(syms)):
        yield dict(zip(syms, values))

def conjoin(sentences):
    result = sentences[0]
    for s in sentences[1:]:
        result = And(result, s)
    return result

def tt_entails(kb, alpha):
    syms = symbols(kb) | symbols(alpha)
    checked = 0
    for model in all_models(syms):
        checked += 1
        if pl_true(kb, model) and not pl_true(alpha, model):
            return False, checked
    return True, checked

def is_valid(s):
    return all(pl_true(s, m) for m in all_models(symbols(s)))

def is_satisfiable(s):
    return any(pl_true(s, m) for m in all_models(symbols(s)))

def print_truth_table(s):
    syms = sorted(symbols(s))
    header = ' | '.join(syms) + ' | ' + to_str(s)
    print(header)
    print('-' * len(header))
    for model in all_models(syms):
        row = ' | '.join(('T' if model[x] else 'F').ljust(len(x)) for x in syms)
        print(row + ' | ' + ('T' if pl_true(s, model) else 'F'))

In [7]:
print('Modus Ponens  :', tt_entails(And(P, Imp(P, Q)), Q))
print('(P | Q) |= P  :', tt_entails(Or(P, Q), P))
print('Modus Tollens :', tt_entails(And(Not(Q), Imp(P, Q)), Not(P)))
print('P | ~P valid  :', is_valid(Or(P, Not(P))))
print('P & ~P sat    :', is_satisfiable(And(P, Not(P))))

kb3 = And(Imp(P, Q), Imp(Q, R))
print('3 тэмдэгт     :', tt_entails(kb3, Imp(P, R)))
S = Sym('S')
kb4 = And(kb3, Imp(R, S))
print('4 тэмдэгт     :', tt_entails(kb4, Imp(P, S)))

Modus Ponens  : (True, 4)
(P | Q) |= P  : (False, 3)
Modus Tollens : (True, 4)
P | ~P valid  : True
P & ~P sat    : False
3 тэмдэгт     : (True, 8)
4 тэмдэгт     : (True, 16)


In [8]:
left = Not(And(P, Q))
right = Or(Not(P), Not(Q))
print_truth_table(left)
print()
print_truth_table(right)
print()
print('Де Морган:', all(pl_true(left, m) == pl_true(right, m) for m in all_models({'P', 'Q'})))

P | Q | ~(P & Q)
----------------
T | T | F
T | F | T
F | T | T
F | F | T

P | Q | (~P | ~Q)
-----------------
T | T | F
T | F | T
F | T | T
F | F | T

Де Морган: True


## Алхам 4: CNF-д хөрвүүлэх

In [9]:
def eliminate_iff(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'not':
        return Not(eliminate_iff(s[1]))
    a = eliminate_iff(s[1])
    b = eliminate_iff(s[2])
    if op == 'iff':
        return And(Imp(a, b), Imp(b, a))
    return (op, a, b)

def eliminate_imp(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'not':
        return Not(eliminate_imp(s[1]))
    a = eliminate_imp(s[1])
    b = eliminate_imp(s[2])
    if op == 'imp':
        return Or(Not(a), b)
    return (op, a, b)

def push_not(s):
    op = s[0]
    if op == 'sym':
        return s
    if op == 'not':
        inner = s[1]
        if inner[0] == 'sym':
            return s
        if inner[0] == 'not':
            return push_not(inner[1])
        if inner[0] == 'and':
            return Or(push_not(Not(inner[1])), push_not(Not(inner[2])))
        return And(push_not(Not(inner[1])), push_not(Not(inner[2])))
    return (op, push_not(s[1]), push_not(s[2]))

def distribute(s):
    op = s[0]
    if op == 'and':
        return And(distribute(s[1]), distribute(s[2]))
    if op == 'or':
        a = distribute(s[1])
        b = distribute(s[2])
        if a[0] == 'and':
            return And(distribute(Or(a[1], b)), distribute(Or(a[2], b)))
        if b[0] == 'and':
            return And(distribute(Or(a, b[1])), distribute(Or(a, b[2])))
        return Or(a, b)
    return s

def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))

def negate_literal(lit):
    return lit[1:] if lit.startswith('~') else '~' + lit

def literal_of(s):
    if s[0] == 'sym':
        return s[1]
    return '~' + s[1][1]

def collect(s, op):
    if s[0] == op:
        return collect(s[1], op) + collect(s[2], op)
    return [s]

def to_clauses(s):
    clauses = set()
    for conjunct in collect(s, 'and'):
        literals = frozenset(literal_of(x) for x in collect(conjunct, 'or'))
        if any(negate_literal(l) in literals for l in literals):
            continue
        clauses.add(literals)
    return clauses

def clause_str(c):
    return '{' + ', '.join(sorted(c)) + '}' if c else '{}'

In [10]:
A, B, C = Sym('A'), Sym('B'), Sym('C')
bc = Iff(B, Or(A, C))
print(to_str(to_cnf(bc)))
for c in sorted(to_clauses(to_cnf(bc)), key=sorted):
    print(clause_str(c))

def equivalent(a, b):
    syms = symbols(a) | symbols(b)
    return all(pl_true(a, m) == pl_true(b, m) for m in all_models(syms))

tests = [bc, Imp(And(P, Q), R), Not(Iff(P, Q)), Imp(Or(P, Q), And(R, Not(P)))]
for t in tests:
    print(to_str(t), '->', equivalent(t, to_cnf(t)))

((~B | (A | C)) & ((~A | B) & (~C | B)))
{A, C, ~B}
{B, ~A}
{B, ~C}
(B <=> (A | C)) -> True
((P & Q) => R) -> True
~(P <=> Q) -> True
((P | Q) => (R & ~P)) -> True


## Алхам 5: Резолюц

In [11]:
RESOLVENTS = 0

def resolve(ci, cj):
    result = []
    for lit in ci:
        neg = negate_literal(lit)
        if neg in cj:
            merged = (ci - {lit}) | (cj - {neg})
            if any(negate_literal(l) in merged for l in merged):
                continue
            result.append(frozenset(merged))
    return result

def resolution_unsat(clauses, limit=100000):
    global RESOLVENTS
    RESOLVENTS = 0
    clauses = set(clauses)
    while True:
        pool = sorted(clauses, key=sorted)
        new = set()
        for i in range(len(pool)):
            for j in range(i + 1, len(pool)):
                for r in resolve(pool[i], pool[j]):
                    RESOLVENTS += 1
                    if not r:
                        return True
                    if RESOLVENTS > limit:
                        return None
                    new.add(r)
        if new <= clauses:
            return False
        clauses |= new

def pl_resolution(kb_clauses, alpha, limit=100000):
    negated = to_clauses(to_cnf(Not(alpha)))
    return resolution_unsat(set(kb_clauses) | negated, limit)

In [12]:
print(resolve(frozenset({'A', 'B'}), frozenset({'~A', 'C'})))
print(resolve(frozenset({'C'}), frozenset({'~C'})))

kb_abc = conjoin([Or(A, B), Or(Not(A), C), Or(Not(B), C)])
kb_abc_clauses = to_clauses(to_cnf(kb_abc))
print('KB |= C :', pl_resolution(kb_abc_clauses, C))
print('KB |= A :', pl_resolution(kb_abc_clauses, A))

alphas = [C, A, B, Or(A, B), And(A, C), Imp(A, C), Not(C)]
for alpha in alphas:
    by_res = pl_resolution(kb_abc_clauses, alpha)
    by_tt = tt_entails(kb_abc, alpha)[0]
    print(to_str(alpha), by_res, by_tt, by_res == by_tt)

kb_mp = conjoin([Imp(P, Q), P])
kb_mp_clauses = to_clauses(to_cnf(kb_mp))
for alpha in [Q, Not(Q), P, R]:
    print(to_str(alpha), pl_resolution(kb_mp_clauses, alpha), tt_entails(kb_mp, alpha)[0])

[frozenset({'B', 'C'})]
[frozenset()]
KB |= C : True
KB |= A : False
C True True True
A False False True
B False False True
(A | B) True True True
(A & C) False False True
(A => C) True True True
~C False False True
Q True True
~Q False False
P True True
R False False


## Алхам 6: DPLL

In [13]:
DPLL_CALLS = 0

def lit_value(lit, model):
    if lit.startswith('~'):
        name = lit[1:]
        if name not in model:
            return None
        return not model[name]
    if lit not in model:
        return None
    return model[lit]

def find_pure_symbol(syms, clauses, model):
    active = [c for c in clauses if True not in [lit_value(l, model) for l in c]]
    for s in syms:
        positive = any(s in c for c in active)
        negative = any('~' + s in c for c in active)
        if positive != negative:
            return s, positive
    return None

def find_unit_clause(clauses, model):
    for c in clauses:
        values = [lit_value(l, model) for l in c]
        if True in values:
            continue
        free = [l for l, v in zip(c, values) if v is None]
        if len(free) == 1:
            lit = free[0]
            if lit.startswith('~'):
                return lit[1:], False
            return lit, True
    return None

def dpll(clauses, syms, model, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS += 1
    undecided = False
    for c in clauses:
        values = [lit_value(l, model) for l in c]
        if True in values:
            continue
        if None not in values:
            return False
        undecided = True
    if not undecided:
        return model
    if use_pure:
        found = find_pure_symbol(syms, clauses, model)
        if found:
            s, v = found
            rest = [x for x in syms if x != s]
            return dpll(clauses, rest, {**model, s: v}, use_pure, use_unit)
    if use_unit:
        found = find_unit_clause(clauses, model)
        if found:
            s, v = found
            rest = [x for x in syms if x != s]
            return dpll(clauses, rest, {**model, s: v}, use_pure, use_unit)
    s = syms[0]
    rest = syms[1:]
    result = dpll(clauses, rest, {**model, s: True}, use_pure, use_unit)
    if result is not False:
        return result
    return dpll(clauses, rest, {**model, s: False}, use_pure, use_unit)

def dpll_satisfiable(clauses, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS = 0
    syms = sorted({l.lstrip('~') for c in clauses for l in c})
    ordered = sorted(clauses, key=sorted)
    return dpll(ordered, syms, {}, use_pure, use_unit)

def dpll_entails(kb_clauses, alpha):
    negated = to_clauses(to_cnf(Not(alpha)))
    return dpll_satisfiable(set(kb_clauses) | negated) is False

In [14]:
def cl(*lits):
    return frozenset(lits)

sat_set = [cl('A'), cl('~A', 'B'), cl('~B', 'C'), cl('~C', 'D', 'E'), cl('~D', '~E')]
model = dpll_satisfiable(sat_set)
print(model, 'дуудалт:', DPLL_CALLS)
print('D, E-ийн яг нэг нь үнэн:', model['D'] != model['E'])

unsat_set = [cl('A', 'B'), cl('~A', 'B'), cl('A', '~B'), cl('~A', '~B')]
print(dpll_satisfiable(unsat_set), 'дуудалт:', DPLL_CALLS)

print('KB |= C :', dpll_entails(kb_abc_clauses, C))
print('KB |= A :', dpll_entails(kb_abc_clauses, A))

{'A': True, 'B': True, 'C': True, 'D': True, 'E': False} дуудалт: 6
D, E-ийн яг нэг нь үнэн: True
False дуудалт: 5
KB |= C : True
KB |= A : False


## Алхам 7: Wumpus-ийн мэдлэгийн сан

In [15]:
P11, P12, P21, P22, P31 = [Sym(n) for n in ['P11', 'P12', 'P21', 'P22', 'P31']]
B11, B21 = Sym('B11'), Sym('B21')

R1 = Not(P11)
R2 = Iff(B11, Or(P12, P21))
R3 = Iff(B21, Or(Or(P11, P22), P31))
R4 = Not(B11)
R5 = B21

wumpus_kb = conjoin([R1, R2, R3, R4, R5])
wumpus_clauses = to_clauses(to_cnf(wumpus_kb))

for c in sorted(wumpus_clauses, key=sorted):
    print(clause_str(c))

kb_models = [m for m in all_models(symbols(wumpus_kb)) if pl_true(wumpus_kb, m)]
print('KB-ийг хангах загвар:', len(kb_models))

{B11, ~P12}
{B11, ~P21}
{B21}
{B21, ~P11}
{B21, ~P22}
{B21, ~P31}
{P11, P22, P31, ~B21}
{P12, P21, ~B11}
{~B11}
{~P11}
KB-ийг хангах загвар: 3


In [16]:
queries = [
    ('~P12', Not(P12)),
    ('~P21', Not(P21)),
    ('~P22', Not(P22)),
    ('~P31', Not(P31)),
    ('P22 | P31', Or(P22, P31)),
]

def ask_three(kb, kb_clauses, alpha):
    tt, checked = tt_entails(kb, alpha)
    res = pl_resolution(kb_clauses, alpha)
    res_count = RESOLVENTS
    dp = dpll_entails(kb_clauses, alpha)
    return tt, checked, res, res_count, dp, DPLL_CALLS

print('Асуулт      | TT      | Резолюц   | DPLL')
for name, alpha in queries:
    tt, checked, res, res_count, dp, calls = ask_three(wumpus_kb, wumpus_clauses, alpha)
    agree = tt == res == dp
    print(name.ljust(11), '|', str(tt).ljust(5), checked, '|', str(res).ljust(5), res_count, '|', str(dp).ljust(5), calls, '| ижил:', agree)

Асуулт      | TT      | Резолюц   | DPLL
~P12        | True  128 | True  7 | True  7 | ижил: True
~P21        | True  128 | True  7 | True  7 | ижил: True
~P22        | False 93 | False 127 | False 7 | ижил: True
~P31        | False 93 | False 127 | False 8 | ижил: True
P22 | P31   | True  128 | True  172 | True  8 | ижил: True


## Алхам 8: Харьцуулалт

In [17]:
def random_3sat(n, m, seed):
    random.seed(seed)
    clauses = []
    for _ in range(m):
        chosen = random.sample(range(1, n + 1), 3)
        lits = []
        for v in chosen:
            name = 'x' + str(v)
            lits.append(name if random.choice([True, False]) else '~' + name)
        clauses.append(frozenset(lits))
    return clauses

def tt_satisfiable(clauses, limit=200000):
    syms = sorted({l.lstrip('~') for c in clauses for l in c})
    checked = 0
    for model in all_models(syms):
        checked += 1
        if checked > limit:
            return None, limit
        if all(any(lit_value(l, model) for l in c) for c in clauses):
            return True, checked
    return False, checked

def cell(value, count):
    if value is None:
        return 'хэт удаан'
    return str(count)

In [18]:
rows = []

for name, alpha in [('Wumpus: ~P12', Not(P12)), ('Wumpus: ~P22', Not(P22)), ('Wumpus: P22 | P31', Or(P22, P31))]:
    tt, checked = tt_entails(wumpus_kb, alpha)
    res = pl_resolution(wumpus_clauses, alpha)
    res_count = RESOLVENTS
    dp = dpll_entails(wumpus_clauses, alpha)
    rows.append((name, str(tt), cell(tt, checked), str(res), cell(res, res_count), str(dp), cell(dp, DPLL_CALLS)))

for n in (10, 15, 20):
    clauses = random_3sat(n, 4 * n, 1)
    sat_tt, checked = tt_satisfiable(clauses)
    unsat = resolution_unsat(clauses)
    res_count = RESOLVENTS
    model = dpll_satisfiable(clauses)
    sat_dpll = model is not False
    sat_res = None if unsat is None else not unsat
    rows.append(('3-SAT, n = ' + str(n), str(sat_tt), cell(sat_tt, checked), str(sat_res), cell(unsat, res_count), str(sat_dpll), str(DPLL_CALLS)))

print('Бодлого'.ljust(20), 'ТХ хариу'.ljust(9), 'ТХ загвар'.ljust(10), 'Рез. хариу'.ljust(11), 'Resolvent'.ljust(11), 'DPLL хариу'.ljust(11), 'Дуудалт')
for r in rows:
    print(r[0].ljust(20), r[1].ljust(9), r[2].ljust(10), r[3].ljust(11), r[4].ljust(11), r[5].ljust(11), r[6])

Бодлого              ТХ хариу  ТХ загвар  Рез. хариу  Resolvent   DPLL хариу  Дуудалт
Wumpus: ~P12         True      128        True        7           True        7
Wumpus: ~P22         False     93         False       127         False       7
Wumpus: P22 | P31    True      128        True        172         True        8
3-SAT, n = 10        True      18         None        хэт удаан   True        11
3-SAT, n = 15        True      987        None        хэт удаан   True        16
3-SAT, n = 20        True      149663     None        хэт удаан   True        21


In [19]:
print('Бодлого'.ljust(14), 'бүгд'.ljust(8), 'pure-гүй'.ljust(10), 'unit-гүй'.ljust(10), 'хоёулаа үгүй')
for n in (10, 15, 20):
    clauses = random_3sat(n, 4 * n, 1)
    counts = []
    for use_pure, use_unit in [(True, True), (False, True), (True, False), (False, False)]:
        dpll_satisfiable(clauses, use_pure, use_unit)
        counts.append(DPLL_CALLS)
    print(('n = ' + str(n)).ljust(14), str(counts[0]).ljust(8), str(counts[1]).ljust(10), str(counts[2]).ljust(10), counts[3])

Бодлого        бүгд     pure-гүй   unit-гүй   хоёулаа үгүй
n = 10         11       10         12         13
n = 15         16       16         84         161
n = 20         21       19         141        257
